# Adaptive importance sampling for ridge-shaped tr-PL posteriors: tempered SMC and AMIS

Module `inference/smc.py` (15/10/2026).  Input: a finished `main.ipynb` run (mode `trpl`, `sspl` or `joint`): data,
best fit θ* with its Laplace covariance (`trpl`: `curve_fit` covariance of the ODE fit; `sspl`/`joint`: Gauss–Newton
covariance at θ* with the σ of this notebook — exact ss-PL model, tr-PL with the ODE), and — if network, prior and σ
agree — its NUTS samples and full-box grid as references.

1. **Setup**: network, prior, σ, likelihood; one batched evaluator for the whole particle population (one XLA
   program, compiled before any timing).
2. **Tempered SMC**: prior → posterior through adaptive β steps (ESS ≈ 0.5 N per step), systematic resampling,
   random-walk Metropolis moves with the covariance of the weighted particles.  Independent islands
   (= "chains" for R-hat, MC error of log Z).  Gives log Z and, from the β schedule, log Z for every σ' ≥ σ.
3. **Fixed cylinder IS** (cylinder.py, at this σ) and **AMIS**: Student-t mixture proposal in the unconstrained
   coordinates u, initialised from the eigenvector cylinders + 7.5 % defensive prior draws, refitted by weighted EM,
   balance-heuristic weights over all past draws, PSIS k̂.
4. **NUTS reference** (samples of the main run, or re-run here when network/prior/σ differ).
5. **Comparison**: marginals (+ grid), coverage of the NUTS mass (cylinders, AMIS proposal), classifier two-sample
   test, ESS, evaluations and wall time.
6. **Model comparison** (optional): log Z of a second model (other number of traps) on the same data and σ.
7. **HDF5 corner export** (schema of `cylinder.ipynb`, Origin script / `corner_plot`) and previews.
8. **Timing table** (timeit, repeats of the T8 timing config) and summary.

Every likelihood evaluation of SMC and AMIS is one batched surrogate call for the whole population.

In [ ]:
# ── Which run ────────────────────────────────────────────────────────────────
CONFIG_PATH = None          # None = config snapshot of RUN_IN (settings.json); a path overrides it
RUN_IN = "results/2trap_85-15_film/2trap_85-15_film_joint"   # finished main.ipynb run (mode trpl, sspl or joint)
TIMING_CONFIG = "configs/2trap_85-15_film_inference.json" # timeit repeats: timing.repeats_long for the samplers
SMC_REPEATS = None          # timeit repeats of SMC, AMIS and the cylinder IS (None = timing.repeats_long of TIMING_CONFIG)
UPDATE_T8 = True            # add the SMC row to the T8 table of RUN_IN (only if network, prior and sigma are those of RUN_IN)

# ── Network and prior (None = those of the config) ──────────────────────────
NN_DIR = None               # e.g. "networks/2trap"
NN_STEM = None              # file stem of training HDF5 / scalers (None = config, or from the scaler file)
PRIOR_OVERRIDE = None       # {"Et_rel_1": [0.5, 0.98], ...}: replaces/extends the config bounds_override

# ── sigma of the tr-PL term ─────────────────────────────────────────────────
# "run" = sigma_eff of RUN_IN (the sigma of its NUTS and grid); or a cylinder.sigma_new spec,
# e.g. {"kind": "base", "base": 0.49361} (= sqrt(base^2 + fit_MSE_ODE), the sigma of cylinder.ipynb).
# The ss-PL term (modes sspl / joint) always uses the sigma_eff of RUN_IN.
SIGMA_SPEC = "run"

# ── References ──────────────────────────────────────────────────────────────
NUTS_REFERENCE = "run_in"   # "run_in" = NUTS samples of RUN_IN (needs the same network, prior and sigma) | "run"
                            # | None = no NUTS reference (SMC is the posterior; no comparison sections)
NUTS_REPEATS = 1            # timeit repeats if NUTS is run here (each ~10 min)
EXTRA_NUTS = None           # {"label": ..., "path": ...}: other NUTS samples, only for the coverage table

# ── SMC / AMIS / cylinders ──────────────────────────────────────────────────
SMC = dict(n_particles=4096, n_islands=4, ess_target=0.5, target_accept=0.25, p_move=0.95, min_mh=2, max_mh=30)
RUN_AMIS = True             # needs the cylinders (RUN_CYL_IS)
RUN_CYL_IS = True           # fixed eigenvector-cylinder IS
AMIS = dict(n_per_iter=16384, n_iter=10, alpha_def=0.075, nu=5.0, cyl_m=13, n_split=3, cov_inflation=1.5, birth_threshold=20.0)
CYL = dict(radius=2.0, sigma_min=0.02)
N_JOBS = 6                  # CPU workers for the ODE solves of the Gauss-Newton covariance (mode joint)

# ── Model comparison (None = skip): a second model on the same data and sigma ──
COMPARE = None              # {"label": "2 traps (net 3611)", "nn_dir": ..., "nn_stem": ..., "bounds_override": {...}}
LABEL = None                # label of the model of this run (None = "<n> trap(s) (net <id>)")

MAX_BATCH = 16384           # rows per surrogate call (fixed shape: one XLA program); >= the SMC population
SEED = 20261015
OUT_TAG = None              # output folder name (None = smc_<network>_sigma<sigma>)
CUDA_DEVICE = "1"

## 0. Imports, settings, output folder

In [ ]:
import os, json, math, warnings
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = str(CUDA_DEVICE)
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
%matplotlib inline
warnings.filterwarnings("ignore", category=UserWarning)

from inference import surrogate, likelihood, samplers, report, smc, runs, cylinder as cy, corner_plot as cp
from inference import benchmark as bm
from inference.prior import Prior
from inference.benchmark import timeit_call, format_mean_sd
surrogate.configure_tensorflow()

RUN_IN = Path(RUN_IN)
settings_in = json.loads((RUN_IN / "settings.json").read_text())
cfg = json.loads(Path(CONFIG_PATH).read_text()) if CONFIG_PATH else settings_in["config"]
MODE = settings_in["mode"]                 # 'trpl' | 'sspl' | 'joint'
tcfg = json.loads(Path(TIMING_CONFIG).read_text())["timing"] if TIMING_CONFIG else cfg["timing"]
REPEATS, REPEATS_LONG = int(tcfg["repeats"]), int(SMC_REPEATS or tcfg["repeats_long"])
best = json.loads((RUN_IN / "6_best_fit.json").read_text())
sampling_in = json.loads((RUN_IN / "sampling_info.json").read_text()) if (RUN_IN / "sampling_info.json").exists() else {}
Eg, npulse, T = cfg["physics"]["Eg_eV"], cfg["physics"]["npulse_cm3"], cfg["physics"]["T_K"]

def resolve_net(nn_dir, nn_stem):
    nn_dir = Path(nn_dir)
    if nn_stem is None and not (nn_dir / "surrogate_meta.json").exists():
        nn_stem = sorted(nn_dir.glob("*_output_scaler.joblib"))[0].name.replace("_output_scaler.joblib", "")
    return nn_dir, nn_stem
nn_dir, nn_stem = resolve_net(NN_DIR or cfg["nn"]["dir"], NN_STEM if NN_DIR else (NN_STEM or cfg["nn"].get("stem")))
net_id = lambda p: p.name.split("-")[2] if p.name[:8].isdigit() else p.name

TIMINGS = []
def log_timing(t, n_evals=None, note="", method=None):
    row = t.as_dict(); row.update(n_evals=n_evals, note=note, method=method); TIMINGS.append(row)
    print(t, f"| evals: {n_evals:,}" if n_evals else "")
print(f"run: {RUN_IN} | mode: {MODE} | network: {nn_dir.name} | timeit repeats (samplers): {REPEATS_LONG}")

## 1. Network, prior, data, likelihood, batched evaluator

The prior is the training box of the network ∩ the config bounds (∩ `PRIOR_OVERRIDE`).  The references of `RUN_IN`
(NUTS, grid) are used only if network, prior box and σ are the same as here.

In [ ]:
art = surrogate.load_network(nn_dir, nn_stem)
bounds = dict(cfg["prior"]["bounds_override"]); bounds.update(PRIOR_OVERRIDE or {})
def make_prior(art_, bounds_, fixed_=None):
    return Prior.from_artifacts(art_, Eg=Eg, order_traps=cfg["prior"]["order_traps"],
                                photodoping_traps=cfg["prior"]["photodoping_traps"], bounds_override=bounds_,
                                fixed=fixed_ if fixed_ is not None else cfg["prior"].get("fixed", {}), T=T)
prior = make_prior(art, bounds)
N_TRAPS, D = prior.n_traps, prior.dim
free = list(prior.free_idx)
keys = prior.display_keys
trpl = pd.read_csv(RUN_IN / "trpl_used.csv")
sig_run = runs.run_sigmas(RUN_IN)
sigma_ss = sig_run["sspl"] if MODE in ("sspl", "joint") else None
if SIGMA_SPEC == "run":
    sigma = float(sig_run["trpl"]); sig_info = {"kind": "run (sigma_eff of RUN_IN)", "sigma_trpl": sigma}
else:
    sig_info = cy.sigma_new(SIGMA_SPEC, RUN_IN, nn_dir); sigma = float(sig_info["sigma_trpl"])
sig_info["sigma_sspl"] = sigma_ss
def make_log_post(art_, prior_):
    return runs.make_log_post(cfg, RUN_IN, art_, prior_, MODE, sigma, sigma_ss)
log_post, sur = make_log_post(art, prior)
LABEL = LABEL or f"{N_TRAPS} trap{'s' if N_TRAPS > 1 else ''} (net {net_id(nn_dir)})"

theta_star = np.array(best["theta_best"], float)
if not prior.in_support(theta_star[None])[0]:
    print("NOTE: the ODE best fit theta* lies outside this prior: projected into the support")
    theta_star = prior.project(theta_star[None])[0]
truth_theta = None
if cfg.get("true_params"):
    tp = cfg["true_params"]
    truth_theta = np.atleast_2d(prior.from_physical(tp["Et_eV"], tp["Nt_cm3"], tp["tau_n_s"], tp["tau_p_s"], tp["krad_cm3s"]))[0]

# are the references of RUN_IN valid here?
pb_in = pd.read_csv(RUN_IN / "prior_bounds.csv").set_index("parameter")
same_net = Path(settings_in["config"]["nn"]["dir"]).resolve() == nn_dir.resolve()
same_prior = np.allclose(pb_in.loc[prior.names, "theta_lo"], prior.lo) and np.allclose(pb_in.loc[prior.names, "theta_hi"], prior.hi)
same_sigma = (MODE == "sspl" or abs(float(sig_run["trpl"]) - sigma) < 1e-9) and \
             (sigma_ss is None or abs(float(sig_run["sspl"]) - sigma_ss) < 1e-9)
REF_OK = bool(same_net and same_prior and same_sigma)
if NUTS_REFERENCE == "run_in" and not REF_OK:
    print("NUTS/grid of RUN_IN do not match (network/prior/sigma) -> NUTS is run here")
    NUTS_REFERENCE = "run"

OUT = RUN_IN / (OUT_TAG or (f"smc_{net_id(nn_dir)}_sigma{sigma:.4f}" if MODE != "sspl" else "smc_sspl"))
FIG = OUT / "figures"; FIG.mkdir(parents=True, exist_ok=True)
report.save_json({"run_in": str(RUN_IN), "mode": MODE, "network": str(nn_dir), "stem": nn_stem, "sigma": sig_info,
                  "prior_override": PRIOR_OVERRIDE, "smc": SMC, "amis": AMIS, "cylinders": CYL, "compare": COMPARE,
                  "nuts_reference": NUTS_REFERENCE, "max_batch": MAX_BATCH, "seed": SEED,
                  "timing": {"repeats": REPEATS, "repeats_long": REPEATS_LONG, "config": TIMING_CONFIG},
                  "references_of_run_in_valid": {"network": bool(same_net), "prior": bool(same_prior), "sigma": bool(same_sigma)},
                  "config": cfg}, OUT / "settings.json")
pr_tab = prior.describe(); pr_tab.to_csv(OUT / "1_prior_bounds.csv", index=False)

# one evaluator (fixed shape -> one XLA program) for SMC and AMIS; compilation is timed separately
ev = smc.PopulationEvaluator(log_post, max_batch=MAX_BATCH, fixed_shape=True)
_, t_comp = timeit_call(ev.precompile, [MAX_BATCH], repeats=1, label="XLA compile of the batched evaluator (untimed elsewhere)")
log_timing(t_comp, note=f"one program, {MAX_BATCH} rows")
print(f"model: {LABEL} | mode {MODE} | sigma_trpl = {sigma:.5f} ({sig_info['kind']}), sigma_sspl = {sigma_ss} | "
      f"tr-PL points: {sur.n_points if sur is not None else 0} | "
      f"references of RUN_IN valid: {REF_OK} (net {same_net}, prior {same_prior}, sigma {same_sigma})")
print("output:", OUT)

panels = (["trpl"] if sur is not None else []) + (["sspl"] if sigma_ss is not None else [])
fig, axs = plt.subplots(1, len(panels), figsize=(5 * len(panels), 3.2), squeeze=False)
for ax, what in zip(axs[0], panels):
    if what == "trpl":
        pred = log_post.predict_trpl(theta_star)
        ax.plot(trpl.qfls, np.log10(trpl.tau_diff), "k.", ms=3, label="data")
        ax.plot(pred["qfls_used"], pred["log10_tau_used"][0], "C3-", label="NN at the best fit θ*")
        ax.fill_between(pred["qfls_used"], pred["log10_tau_obs"] - sigma, pred["log10_tau_obs"] + sigma, color="0.85", label="±σ")
        ax.set_xlabel("QFLS (eV)"); ax.set_ylabel(r"$\log_{10}\tau_\mathrm{diff}$ (s)")
    else:
        ps = log_post.predict_sspl(theta_star)
        ax.plot(ps["x_exp"], ps["y_obs"], "k.", ms=3, label="data")
        ax.plot(ps["x_exp"], ps["y_model"][0], "C3-", label="ss-PL model at θ*")
        ax.fill_between(ps["x_exp"], ps["y_obs"] - sigma_ss, ps["y_obs"] + sigma_ss, color="0.85", label="±σ")
        ax.set_xlabel(log_post.sspl_term.observation.mode.split("_vs_")[1]); ax.set_ylabel(log_post.sspl_term.observation.mode.split("_vs_")[0])
    ax.legend(fontsize=7)
fig.tight_layout(); report.save_figure(fig, FIG / "1_data_best_fit.png"); plt.show()
pr_tab

## 2. Likelihood-tempered SMC

`run_smc` is timed with `timeit` (`timing.repeats_long` repeats, same seed → identical runs).  The table lists every
stage of every island: β, ESS after reweighting, MH moves and acceptance, particle diversity, evaluations, time.

In [ ]:
smc_res, t_smc = timeit_call(smc.run_smc, log_post, **SMC, seed=SEED, evaluator=ev, verbose=False,
                             repeats=REPEATS_LONG, label="tempered SMC")
si = smc_res.info
log_timing(t_smc, n_evals=si["evals_total_points"], method="SMC",
           note=f"{si['n_islands']} islands x {si['n_particles']} particles, stages {si['n_stages']}, {si['evals_total_calls']} batched calls")
diag_smc = samplers.diagnostics(smc_res, prior)
print(f"log Z = {smc_res.log_evidence:.3f} ± {smc_res.log_evidence_se:.3f} (islands {np.round(si['log_evidence_islands'], 3)}) | "
      f"Kish ESS (weighted, beta = 1) = {si['ess_weighted_kish']:.0f} | distinct final particles {si['n_unique_final']} | "
      f"max R-hat (islands) {diag_smc.r_hat.max():.4f} | min ESS bulk {diag_smc.ess_bulk.min():.0f} | k̂ of the last reweighting {si['k_hat_last_reweighting']:.2f}")
smc_res.stages.to_csv(OUT / "2_smc_stages.csv", index=False)
report.save_json(si, OUT / "2_smc_info.json")
report.export_samples(smc_res, prior, OUT / "posterior_samples_smc.csv")
diag_smc.to_csv(OUT / "diagnostics_smc.csv")
th_w, w_w = smc_res.flat_weighted()
pd.concat([pd.DataFrame(th_w, columns=prior.names), pd.DataFrame({"weight": w_w, "island": smc_res.weighted["group"]})],
          axis=1).to_csv(OUT / "2_smc_weighted_samples.csv", index=False, float_format="%.8g")
sp = smc.sigma_path(smc_res, sigma); sp.to_csv(OUT / "2_smc_sigma_path.csv", index=False)
# credible intervals of the SMC posterior (weighted particles resampled to 20000 equal-weight draws)
smc_eq = smc.resample_weighted(th_w, w_w, 20000, seed=1)
ci_smc = report.credible_intervals(prior.to_display(smc_eq), prior.display_bounds(), keys,
                                  truth=None if truth_theta is None else prior.to_display(truth_theta[None])[0],
                                  best=prior.to_display(theta_star[None])[0])
ci_smc.to_csv(OUT / "credible_intervals_smc.csv")
ci_smc_derived = report.derived_intervals(smc_eq, prior); ci_smc_derived.to_csv(OUT / "credible_intervals_derived_smc.csv")
display(ci_smc)
display(smc_res.stages.round(4)); diag_smc

In [ ]:
st = smc_res.stages
fig, axs = plt.subplots(2, 3, figsize=(12, 6))
for i, g in st.groupby("island"):
    axs[0, 0].semilogy(g.stage, g.beta, "o-", ms=3, label=f"island {i}")
    axs[0, 1].plot(g.stage, g.ess_frac, "o-", ms=3)
    axs[0, 2].plot(g.stage, g.acceptance, "o-", ms=3); axs[0, 2].plot(g.stage, g.n_mh / g.n_mh.max(), "x:", ms=3, color="0.5")
    axs[1, 0].plot(g.stage, g.log_z, "o-", ms=3)
    axs[1, 1].plot(g.stage, g.n_unique_after_move / si["n_particles"], "o-", ms=3)
axs[0, 0].set_ylabel("β"); axs[0, 0].legend(fontsize=7)
axs[0, 1].axhline(SMC["ess_target"], color="0.5", ls=":"); axs[0, 1].set_ylabel("ESS / N after reweighting")
axs[0, 2].set_ylabel("MH acceptance (x: moves / max)")
axs[1, 0].set_ylabel("cumulative log Z")
axs[1, 1].set_ylabel("distinct particles / N after moves")
for i, g in sp.groupby("island"):
    axs[1, 2].semilogx(g.sigma, g.log_z_sigma, "o-", ms=3)
axs[1, 2].axvline(sigma, color="k", ls=":"); axs[1, 2].set_xlabel("σ' = σ/√β"); axs[1, 2].set_ylabel("log Z(σ')")
for a in axs.ravel()[:5]:
    a.set_xlabel("stage"); a.grid(alpha=0.3)
fig.suptitle(f"tempered SMC — {LABEL}, σ = {sigma:.4f}", fontsize=10)
fig.tight_layout(); report.save_figure(fig, FIG / "2_smc_adaptation.png"); plt.show()

# particles of island 0 after every stage: marginals from prior (light) to posterior (dark)
H = smc_res.history
fig, axs = plt.subplots(1, len(free), figsize=(2.6 * len(free), 2.4))
cols = plt.cm.viridis(np.linspace(0.05, 0.95, len(H)))
bnd = prior.display_bounds()
for j, i in enumerate(free):
    for h, c in zip(H, cols):
        v = prior.to_display(h["theta"])[:, i]
        axs[j].hist(v, bins=40, range=tuple(bnd[i]), histtype="step", color=c, density=True, lw=1)
    axs[j].set_xlabel(prior.labels[i], fontsize=8); axs[j].set_yticks([])
axs[0].set_title("island 0, stage 0 (prior) → β = 1", fontsize=8, loc="left")
fig.tight_layout(); report.save_figure(fig, FIG / "2_smc_particle_evolution.png"); plt.show()

## 3. Fixed eigenvector-cylinder IS and AMIS

Cylinders (cylinder.py) from the best fit θ* and its Laplace covariance (`trpl`: `curve_fit` covariance rescaled to σ:
(σ/σ_cf)²; `sspl`/`joint`: Gauss–Newton covariance at θ* with the σ of this notebook).  The fixed
cylinder IS (weights L/q) is the baseline; AMIS starts from the same cylinder points (iteration 0) and adapts a
Student-t mixture in the unconstrained coordinates u (+ 7.5 % prior draws in every iteration).

In [ ]:
cyl_res = cyl_summary = cyl = dmap = None
if RUN_CYL_IS:
    if MODE == "trpl" and best.get("cov_theta_curve_fit") is not None:
        dmap, es, cyl = smc.cylinder_setup(prior, theta_star, best["cov_theta_curve_fit"], sigma,
                                           float(best["sigmas"].get("trpl_fit", best["sigmas"]["trpl_base"])), **CYL)
        cov_info = {"method": "curve_fit covariance of main.ipynb x (sigma / sigma_cf)^2"}
    else:
        cov_gn, cov_info = cy.laplace_cov(log_post, theta_star, art, npulse, T, n_jobs=N_JOBS)
        dmap, es, cyl = smc.cylinder_setup(prior, theta_star, cov_gn, 1.0, 1.0, **CYL)
    print("cylinder covariance:", cov_info)
    ll_fn = lambda th: ev(ev.u_free_from_theta(th))[0]
    cyl_res, t_cyl = timeit_call(cy.run, cyl, dmap, prior, theta_star, ll_fn, m=AMIS["cyl_m"], seed=SEED,
                                 repeats=REPEATS_LONG, label=f"fixed cylinder IS (2^{AMIS['cyl_m']} x {es.D})")
    log_timing(t_cyl, n_evals=cyl_res.n_evals, method="cylinder IS")
    from inference.validate import psis
    lw_c = cyl_res.log_w
    k_cyl = float(psis(lw_c[np.isfinite(lw_c)])[1])
    # log Z of the cylinder IS: only the mass INSIDE the cylinders (q = 0 outside)
    px = smc.PriorX(prior, dmap, theta_star)
    logz_cyl = float(np.logaddexp.reduce(lw_c[np.isfinite(lw_c)]) - math.log(len(lw_c)) + px.log_density + np.sum(np.log(es.sigma)))
    cyl_summary = {"covariance": cov_info, "n_evals": cyl_res.n_evals, "ess": cyl_res.ess, "ess_frac": cyl_res.ess / cyl_res.n_evals, "k_hat": k_cyl,
                   "log_z_inside_cylinders": logz_cyl, "sigma_k": es.sigma.tolist(), "chords_z": cyl.chords_z.tolist()}
    report.save_json(cyl_summary, OUT / "3_cylinder_is_summary.json")
    print(json.dumps({k: v for k, v in cyl_summary.items() if not isinstance(v, list)}, indent=1))
else:
    print("cylinder IS skipped (RUN_CYL_IS = False)")

In [ ]:
amis_res = None
if RUN_AMIS and cyl_res is not None:
    amis_res, t_amis = timeit_call(smc.run_amis, log_post, dmap, theta_star, cyl=cyl, **AMIS, seed=SEED, evaluator=ev,
                                   verbose=False, repeats=REPEATS_LONG, label="AMIS (cylinder start)")
    ai = amis_res.info
    log_timing(t_amis, n_evals=ai["evals_total_points"], method="AMIS",
               note=f"{ai['n_iteration0']} cylinder/defensive draws + {AMIS['n_iter']} x {AMIS['n_per_iter']}, {ai['evals_total_calls']} batched calls")
    print(f"AMIS: log Z = {amis_res.log_evidence:.3f} ± {amis_res.log_evidence_se:.3f}, Kish ESS = {amis_res.ess:.0f} "
          f"({100 * amis_res.ess / ai['n_draws_total']:.1f} % of {ai['n_draws_total']:,}), PSIS k̂ = {ai['k_hat']:.2f}, "
          f"{ai['n_births']} component births, final K = {ai['proposals'][-1].K}")
    amis_res.stages.to_csv(OUT / "3_amis_iterations.csv", index=False)
    report.save_json({k: v for k, v in ai.items() if k != "proposals"}, OUT / "3_amis_info.json")
    report.export_samples(amis_res, prior, OUT / "posterior_samples_amis.csv")
    tw, ww = amis_res.flat_weighted()
    pd.concat([pd.DataFrame(tw, columns=prior.names), pd.DataFrame({"weight": ww, "iteration": amis_res.weighted["group"]})],
              axis=1).to_csv(OUT / "3_amis_weighted_samples.csv", index=False, float_format="%.8g")
    mix = ai["proposals"][-1]
    with h5py.File(OUT / "3_amis_proposals.h5", "w") as f:       # every fitted t-mixture (u coordinates)
        for t, m in enumerate(ai["proposals"]):
            g = f.create_group(f"iteration_{t + 1}"); g["weights"], g["means"], g["covs"] = m.weights, m.means, m.covs
            g.attrs["nu"] = m.nu
    st_a = amis_res.stages
    fig, axs = plt.subplots(1, 4, figsize=(14, 3))
    axs[0].errorbar(st_a.iteration, st_a.log_z, st_a.log_z_se, fmt="o-", ms=3, label="AMIS")
    axs[0].axhspan(smc_res.log_evidence - smc_res.log_evidence_se, smc_res.log_evidence + smc_res.log_evidence_se,
                   color="C1", alpha=0.3, label="SMC ± SE")
    axs[0].set_ylabel("log Z"); axs[0].legend(fontsize=7)
    axs[1].plot(st_a.iteration, st_a.ess_frac_total, "o-", ms=3, label="all draws (balance heuristic)")
    axs[1].plot(st_a.iteration, st_a.ess_frac_new_own_proposal, "s--", ms=3, label="new draws vs own q_t")
    axs[1].set_ylabel("ESS / draws"); axs[1].legend(fontsize=7)
    axs[2].plot(st_a.iteration, st_a.k_hat, "o-", ms=3); axs[2].axhline(0.7, color="C3", ls=":"); axs[2].axhline(0.5, color="0.5", ls=":")
    axs[2].set_ylabel("PSIS k̂")
    axs[3].plot(st_a.iteration, st_a.n_components, "o-", ms=3); axs[3].set_ylabel("t components")
    for a in axs:
        a.set_xlabel("iteration"); a.grid(alpha=0.3)
    fig.suptitle(f"AMIS — {LABEL}", fontsize=10); fig.tight_layout(); report.save_figure(fig, FIG / "3_amis_adaptation.png"); plt.show()

    # proposal of iterations 1, mid, last (draws mapped to display coordinates) vs the AMIS posterior
    sel = sorted({0, len(ai["proposals"]) // 2, len(ai["proposals"]) - 1})
    rng_ = np.random.default_rng(0)
    fig, axs = plt.subplots(1, len(free), figsize=(2.6 * len(free), 2.4))
    disp_post = prior.to_display(tw)
    for j, i in enumerate(free):
        axs[j].hist(disp_post[:, i], bins=50, range=tuple(bnd[i]), weights=ww, density=True, color="0.8", label="AMIS posterior")
        for t in sel:
            u_s, _ = ai["proposals"][t].sample(20000, rng_)
            v = prior.to_display(ev.theta(u_s))[:, i]
            axs[j].hist(v, bins=50, range=tuple(bnd[i]), density=True, histtype="step", label=f"proposal it {t + 1}")
        axs[j].set_xlabel(prior.labels[i], fontsize=8); axs[j].set_yticks([])
    axs[0].legend(fontsize=6); fig.tight_layout(); report.save_figure(fig, FIG / "3_amis_proposals.png"); plt.show()
    display(st_a.round(4))

## 4. NUTS reference

`run_in`: the NUTS samples of the main run (same network, prior and σ).  `run`: NUTS with the settings of the config
(32 chains, 1000 warm-up + 2000 draws, dense metric, target_accept 0.95), compiled in a short untimed run first.

In [ ]:
nuts_cfg = cfg["nuts"]
nuts_res = nuts_theta = diag_nuts = nuts_time = None
if NUTS_REFERENCE == "run_in":
    nuts_df = pd.read_csv(RUN_IN / "posterior_samples_nuts.csv")
    nuts_theta = nuts_df[prior.names].values
    C = int(nuts_df.chain.max()) + 1
    nuts_res = samplers.SamplerResult("nuts", theta=nuts_theta.reshape(C, -1, D), u=prior.inverse(nuts_theta).reshape(C, -1, D),
                                      info=sampling_in["nuts"])
    tm = pd.read_csv(RUN_IN / "11_timing.csv")
    row = tm[tm.label.str.startswith("NUTS (warm-up")].iloc[0]
    nuts_time = {"mean_s": float(row.mean_s), "std_s": float(row.std_s), "repeats": int(row.repeats), "source": "RUN_IN 11_timing.csv"}
elif NUTS_REFERENCE == "run":
    rng = np.random.default_rng(nuts_cfg["seed"])
    init_u = samplers.init_dispersed(D, nuts_cfg["n_chains"], rng)
    (_, nuts_sampler), t_nc = timeit_call(samplers.run_nuts, log_post, init_u, n_warmup=1, n_draws=1, metric=nuts_cfg["metric"],
                                          target_accept=nuts_cfg["target_accept"], max_tree_depth=nuts_cfg["max_tree_depth"],
                                          seed=nuts_cfg["seed"], verbose=False, label="NUTS compile (short untimed run)")
    log_timing(t_nc, note="compilation, not part of the NUTS time")
    (nuts_res, _), t_nuts = timeit_call(samplers.run_nuts, log_post, init_u, n_warmup=nuts_cfg["n_warmup"],
                                        n_draws=nuts_cfg["n_draws"], metric=nuts_cfg["metric"],
                                        target_accept=nuts_cfg["target_accept"], max_tree_depth=nuts_cfg["max_tree_depth"],
                                        seed=nuts_cfg["seed"], sampler=nuts_sampler, verbose=False,
                                        repeats=NUTS_REPEATS, label="NUTS (warm-up + sampling)")
    log_timing(t_nuts, n_evals=nuts_res.info["evals_total_points"], method="NUTS",
               note=f"{nuts_cfg['n_chains']} chains x ({nuts_cfg['n_warmup']} + {nuts_cfg['n_draws']}); likelihood + gradient")
    nuts_time = {"mean_s": t_nuts.mean_s, "std_s": t_nuts.std_s, "repeats": len(t_nuts.times_s), "source": "this notebook"}
    report.export_samples(nuts_res, prior, OUT / "posterior_samples_nuts.csv")
    report.save_json(nuts_res.info, OUT / "4_nuts_info.json")
    nuts_theta = nuts_res.flat_theta()
if nuts_res is not None:
    diag_nuts = samplers.diagnostics(nuts_res, prior)
    diag_nuts.to_csv(OUT / "diagnostics_nuts.csv")
    print(f"NUTS ({NUTS_REFERENCE}): {nuts_theta.shape[0]:,} draws, max R-hat {diag_nuts.r_hat.max():.4f}, "
          f"min ESS bulk {diag_nuts.ess_bulk.min():.0f}, {nuts_res.info.get('n_divergent')} divergences, "
          f"time {format_mean_sd(nuts_time['mean_s'], nuts_time['std_s'], nuts_time['repeats'], 'min')}")
    display(diag_nuts)
else:
    print("no NUTS reference (NUTS_REFERENCE = None): SMC is the posterior")

## 5. Comparison: marginals, coverage of the NUTS mass, two-sample test, efficiency

* **Coverage**: fraction of the NUTS samples inside the union of the cylinders (the region the fixed cylinder IS can
  see at all) and inside the 99 % highest-density region of the final AMIS t-mixture.
* **C2ST**: held-out accuracy of a gradient-boosted classifier separating NUTS draws from the draws of a method
  (weighted samples enter with their weights, not resampled: duplicates would be memorised); 0.5 = indistinguishable.
  NUTS chains 1st half vs 2nd half gives the floor of the test.
* **ESS**: SMC — ArviZ bulk ESS over the islands and Kish ESS of the β = 1 weights; AMIS/cylinder IS — Kish ESS of the
  importance weights; NUTS — ArviZ bulk ESS.

In [ ]:
grid_ok = REF_OK and (RUN_IN / "7f_grid_marginals_1d.csv").exists()
sets = {"SMC": (th_w, w_w)}
if nuts_theta is not None:
    sets["NUTS"] = (nuts_theta, None)
if cyl_res is not None:
    sets["cylinder IS"] = (cyl_res.theta, cyl_res.w)
if amis_res is not None:
    sets["AMIS"] = (tw, ww)
marg = smc.marginal_table(sets, prior)
if grid_ok:
    g1 = pd.read_csv(RUN_IN / "7f_grid_marginals_1d.csv")
    for k, g in g1.groupby("parameter"):
        i = keys.index(k)
        if i not in free:                 # fixed parameter (e.g. k_rad): not in the sampler tables
            continue
        v = g.theta.values * (Eg if prior.names[i].startswith("Et_rel") else 1.0)
        q = smc.weighted_quantiles(v, g.mass.values, [0.025, 0.16, 0.5, 0.84, 0.975])
        mean = float(g.mass.values @ v)
        marg = pd.concat([marg, pd.DataFrame([{"method": "grid", "parameter": k, "mean": mean,
                                               "sd": float(np.sqrt(g.mass.values @ (v - mean) ** 2)),
                                               **dict(zip(["q2.5", "q16", "q50", "q84", "q97.5"], q))}])])
marg.to_csv(OUT / "5_marginals.csv", index=False)
# deviation of every method from NUTS in units of the NUTS posterior sd (largest over the parameters)
dev = None
ref = marg[marg.method == "NUTS"].set_index("parameter")
dev_rows = []
for m_, g in (marg.groupby("method") if nuts_theta is not None else []):
    g = g.set_index("parameter")
    d_ = {q: float(((g[q] - ref.loc[g.index, q]).abs() / ref.loc[g.index, "sd"]).max()) for q in ("q2.5", "q16", "q50", "q84", "q97.5")}
    d_["sd ratio (max |1 - sd/sd_NUTS|)"] = float((1 - g["sd"] / ref.loc[g.index, "sd"]).abs().max())
    dev_rows.append({"method": m_, **{f"max |Δ{q}| / sd_NUTS" if q.startswith("q") else q: v for q, v in d_.items()}})
if dev_rows:
    dev = pd.DataFrame(dev_rows).set_index("method")
    dev.to_csv(OUT / "5_quantile_deviation_vs_nuts.csv")

# 1-D marginals
fig, axs = plt.subplots(1, len(free), figsize=(2.9 * len(free), 2.6))
for j, i in enumerate(free):
    r = tuple(bnd[i]); a = axs[j]
    if nuts_theta is not None:
        a.hist(prior.to_display(nuts_theta)[:, i], bins=60, range=r, density=True, color="0.8", label="NUTS")
    a.hist(prior.to_display(th_w)[:, i], bins=60, range=r, weights=w_w, density=True, histtype="step", color="C0", lw=1.4, label="SMC")
    if amis_res is not None:
        a.hist(prior.to_display(tw)[:, i], bins=60, range=r, weights=ww, density=True, histtype="step", color="C1", lw=1.2, ls="--", label="AMIS")
    if cyl_res is not None:
        a.hist(prior.to_display(cyl_res.theta)[:, i], bins=60, range=r, weights=cyl_res.w, density=True, histtype="step", color="C2", lw=0.9, label="cylinder IS")
    if grid_ok:
        g = g1[g1.parameter == keys[i]]
        v = g.theta.values * (Eg if prior.names[i].startswith("Et_rel") else 1.0)
        a.plot(v, g.mass.values / np.gradient(v), "k:", lw=1.2, label="grid")
    if truth_theta is not None:
        a.axvline(prior.to_display(truth_theta[None])[0, i], color="c", lw=1)
    a.set_xlabel(prior.labels[i], fontsize=8); a.set_yticks([])
axs[0].legend(fontsize=6)
fig.suptitle(f"marginal posteriors — {LABEL}, σ = {sigma:.4f}", fontsize=10)
fig.tight_layout(); report.save_figure(fig, FIG / "5_marginals.png"); plt.show()
display(marg.round(4)); dev.round(3) if dev is not None else None

In [ ]:
# corner: NUTS (filled) vs SMC (contours, resampled; smc_eq from section 2)
if nuts_theta is not None:
    fig = report.plot_corner(prior.to_display(nuts_theta), prior,
                             truth=None if truth_theta is None else prior.to_display(truth_theta[None])[0],
                             best=prior.to_display(theta_star[None])[0], overlay=prior.to_display(smc_eq),
                             overlay_label="SMC", label="NUTS")
    report.save_figure(fig, FIG / "5_corner_nuts_vs_smc.png"); plt.show()
    if amis_res is not None:
        amis_eq = smc.resample_weighted(tw, ww, 20000, seed=2)
        fig = report.plot_corner(prior.to_display(nuts_theta), prior,
                                 truth=None if truth_theta is None else prior.to_display(truth_theta[None])[0],
                                 best=prior.to_display(theta_star[None])[0], overlay=prior.to_display(amis_eq),
                                 overlay_label="AMIS", label="NUTS")
        report.save_figure(fig, FIG / "5_corner_nuts_vs_amis.png"); plt.show()

In [ ]:
# coverage of the NUTS mass, two-sample tests (only with a NUTS reference)
cover = c2 = None
if nuts_theta is not None:
    u_nuts = ev.u_free_from_theta(nuts_theta)
    cover = [{"samples": f"NUTS ({NUTS_REFERENCE})", "n": len(nuts_theta),
              "in prior support": float(prior.in_support(nuts_theta).mean()),
              "inside cylinders": cy.fraction_inside(cyl, dmap.to_x(nuts_theta)) if cyl is not None else np.nan,
              "inside AMIS 99 % HDR": smc.mixture_hdr_fraction(amis_res.info["proposals"][-1], u_nuts) if amis_res else np.nan}]
    smc_eq_u = ev.u_free_from_theta(smc_eq)
    cover.append({"samples": "SMC (resampled)", "n": len(smc_eq), "in prior support": 1.0,
                  "inside cylinders": cy.fraction_inside(cyl, dmap.to_x(smc_eq)) if cyl is not None else np.nan,
                  "inside AMIS 99 % HDR": smc.mixture_hdr_fraction(amis_res.info["proposals"][-1], smc_eq_u) if amis_res else np.nan})
    if EXTRA_NUTS:
        th_x = pd.read_csv(EXTRA_NUTS["path"])[prior.names].values
        sup = prior.in_support(th_x)
        cover.append({"samples": EXTRA_NUTS["label"], "n": len(th_x), "in prior support": float(sup.mean()),
                      "inside cylinders": cy.fraction_inside(cyl, dmap.to_x(th_x)) if cyl is not None else np.nan,   # outside the support = outside
                      "inside AMIS 99 % HDR": (smc.mixture_hdr_fraction(amis_res.info["proposals"][-1],
                                               ev.u_free_from_theta(th_x[sup])) * float(sup.mean())) if amis_res else np.nan})
    cover = pd.DataFrame(cover); cover.to_csv(OUT / "5_coverage_of_nuts_mass.csv", index=False)

    Xn = prior.to_display(nuts_theta)[:, free]
    c2 = {"SMC (final particles)": smc.c2st(Xn, prior.to_display(smc_res.flat_theta())[:, free], seed=0),
          "SMC (weighted, beta = 1)": smc.c2st(Xn, prior.to_display(th_w)[:, free], w_b=w_w, seed=0)}
    if cyl_res is not None:
        c2["cylinder IS (weighted)"] = smc.c2st(Xn, prior.to_display(cyl_res.theta)[:, free], w_b=cyl_res.w, seed=0)
    if amis_res is not None:
        c2["AMIS (weighted)"] = smc.c2st(Xn, prior.to_display(tw)[:, free], w_b=ww, seed=0)
    # NUTS vs NUTS (two halves of the chains) = the floor of the test
    half = nuts_res.theta.shape[0] // 2
    c2["NUTS (chains 1st vs 2nd half)"] = smc.c2st(prior.to_display(nuts_res.theta[:half].reshape(-1, D))[:, free],
                                                   prior.to_display(nuts_res.theta[half:].reshape(-1, D))[:, free], seed=0)
    c2 = pd.Series(c2, name="C2ST accuracy vs NUTS"); c2.to_csv(OUT / "5_c2st.csv")
    display(cover.round(4)); display(c2.round(3))

In [ ]:
# efficiency table (one row per method)
eff = []
def row_(method, mean_s, sd_s, n_rep, n_evals, n_calls, ess_kish, ess_bulk, rhat, k_hat, logz, logz_se, note=""):
    ess = ess_bulk if ess_bulk is not None and np.isfinite(ess_bulk) else ess_kish
    return {"method": method, "time_mean_s": mean_s, "time_sd_s": sd_s, "repeats": n_rep, "n_evals": n_evals,
            "n_calls": n_calls, "ess_kish": ess_kish, "min_ess_bulk": ess_bulk, "max_r_hat": rhat, "k_hat": k_hat,
            "ess_per_s": ess / mean_s if (ess is not None and mean_s) else None,
            "ess_per_1k_evals": 1000 * ess / n_evals if (ess is not None and n_evals) else None,
            "log_z": logz, "log_z_se": logz_se, "note": note}
eff.append(row_("SMC", t_smc.mean_s, t_smc.std_s, len(t_smc.times_s), si["evals_total_points"], si["evals_total_calls"],
                si["ess_weighted_kish"], float(diag_smc.ess_bulk.min()), float(diag_smc.r_hat.max()),
                si["k_hat_last_reweighting"], smc_res.log_evidence, smc_res.log_evidence_se,
                "ESS/s from the ArviZ bulk ESS of the final particles"))
if amis_res is not None:
    eff.append(row_("AMIS", t_amis.mean_s, t_amis.std_s, len(t_amis.times_s), ai["evals_total_points"], ai["evals_total_calls"],
                    amis_res.ess, None, None, ai["k_hat"], amis_res.log_evidence, amis_res.log_evidence_se,
                    "ESS = Kish ESS of the balance-heuristic weights"))
if cyl_res is not None:
    eff.append(row_("cylinder IS", t_cyl.mean_s, t_cyl.std_s, len(t_cyl.times_s), cyl_res.n_evals, None, cyl_res.ess, None, None,
                    k_cyl, logz_cyl, None, "log Z only of the mass inside the cylinders"))
if nuts_res is not None:
    ne = bm.sampler_efficiency("nuts", diag_nuts, nuts_res.info, nuts_time["mean_s"], free_keys=[keys[i] for i in free])
    eff.append(row_("NUTS", nuts_time["mean_s"], nuts_time["std_s"], nuts_time["repeats"], nuts_res.info.get("evals_total_points"),
                    nuts_res.info.get("evals_total_calls"), None, ne["min_ess_bulk"], ne["max_r_hat"], None, None, None,
                    f"likelihood + gradient evaluations; {nuts_time['source']}"))
if grid_ok:
    gi = json.loads((RUN_IN / "7f_grid_info.json").read_text())
    tm = pd.read_csv(RUN_IN / "11_timing.csv"); gr = tm[tm.label.str.startswith("grid")].iloc[0]
    eff.append(row_("grid", float(gr.mean_s), float(gr.std_s), int(gr.repeats), int(gi["n_evaluated"]), None, None, None, None, None,
                    gi["log_z"] - math.log(gi["n_points"]), None, f"{gi['points_per_axis']} points per axis; RUN_IN 11_timing.csv"))
eff = pd.DataFrame(eff); eff.to_csv(OUT / "5_efficiency.csv", index=False)
eff.round(4)

## 6. Model comparison (optional): log Z of a second model on the same data and σ

SMC for the second model (its own network and prior), then AMIS started from a t-mixture fitted to the SMC particles.
The σ path of both SMC runs gives the Bayes factor as a function of σ' ≥ σ.

In [ ]:
cmp_tab = None
if COMPARE:
    nn2, stem2 = resolve_net(COMPARE["nn_dir"], COMPARE.get("nn_stem"))
    art2 = surrogate.load_network(nn2, stem2)
    prior2 = make_prior(art2, COMPARE.get("bounds_override", {}), COMPARE.get("fixed", {}))
    log_post2, sur2 = make_log_post(art2, prior2)
    same_points = sur is None or (sur2.n_points == sur.n_points and np.allclose(sur2.qfls_used, sur.qfls_used))
    if sur is not None:
        print(f"{COMPARE['label']}: {sur2.n_points} tr-PL points (this model {sur.n_points}); same points: {same_points}")
    ev2 = smc.PopulationEvaluator(log_post2, max_batch=MAX_BATCH, fixed_shape=True)
    _, t_c2 = timeit_call(ev2.precompile, [MAX_BATCH], repeats=1, label=f"XLA compile ({COMPARE['label']})"); log_timing(t_c2)
    smc2, t_smc2 = timeit_call(smc.run_smc, log_post2, **SMC, seed=SEED + 1, evaluator=ev2, verbose=False,
                               repeats=REPEATS_LONG, label=f"tempered SMC ({COMPARE['label']})")
    log_timing(t_smc2, n_evals=smc2.info["evals_total_points"], method=f"SMC {COMPARE['label']}")
    th2, w2 = smc2.flat_weighted()
    mix2 = smc.fit_tmixture(ev2.u_free_from_theta(th2), w2, n_components=8, nu=AMIS["nu"], seed=SEED)
    dmap2 = cy.DecadeMap(prior2)
    amis2, t_amis2 = timeit_call(smc.run_amis, log_post2, dmap2, th2[np.argmax(w2)], init_mixture=mix2,
                                 **{k: v for k, v in AMIS.items() if k not in ("cyl_m", "n_split")}, seed=SEED + 1,
                                 evaluator=ev2, verbose=False, repeats=REPEATS_LONG, label=f"AMIS ({COMPARE['label']}, SMC start)")
    log_timing(t_amis2, n_evals=amis2.info["evals_total_points"], method=f"AMIS {COMPARE['label']}")
    rows = []
    for lab, s_, a_ in ((LABEL, smc_res, amis_res), (COMPARE["label"], smc2, amis2)):
        rows.append({"model": lab, "log_z_smc": s_.log_evidence, "log_z_smc_se": s_.log_evidence_se,
                     "log_z_amis": a_.log_evidence if a_ is not None else np.nan,
                     "log_z_amis_se": a_.log_evidence_se if a_ is not None else np.nan,
                     "amis_k_hat": a_.info["k_hat"] if a_ is not None else np.nan,
                     "amis_ess": a_.ess if a_ is not None else np.nan, "n_free": s_.info["dim_free"]})
    cmp_tab = pd.DataFrame(rows)
    cmp_tab["log_BF_vs_first (AMIS)"] = cmp_tab.log_z_amis - cmp_tab.log_z_amis.iloc[0]
    cmp_tab["log_BF_vs_first (SMC)"] = cmp_tab.log_z_smc - cmp_tab.log_z_smc.iloc[0]
    cmp_tab["same tr-PL points"] = [True, same_points]
    cmp_tab.to_csv(OUT / "6_model_comparison.csv", index=False)
    report.export_samples(smc2, prior2, OUT / f"6_posterior_samples_smc_{net_id(nn2)}.csv")
    smc2.stages.to_csv(OUT / f"6_smc_stages_{net_id(nn2)}.csv", index=False)
    # Bayes factor vs sigma' from the two tempering schedules: every island interpolated in log sigma' on a
    # common grid (inside the range both schedules cover), then the islands averaged
    sp2 = smc.sigma_path(smc2, sigma)
    sg = np.geomspace(sigma, min(sp.sigma.max(), sp2.sigma.max()), 60)
    def on_grid(spx):
        z = [np.interp(np.log(sg), np.log(g.sigma.values[::-1]), g.log_z_sigma.values[::-1]) for _, g in spx.groupby("island")]
        return np.mean(z, axis=0), np.std(z, axis=0, ddof=1) / np.sqrt(len(z))
    zi1, se1 = on_grid(sp); zi2, se2 = on_grid(sp2)
    pd.DataFrame({"sigma": sg, f"log_z {LABEL}": zi1, f"se {LABEL}": se1, f"log_z {COMPARE['label']}": zi2,
                  f"se {COMPARE['label']}": se2, "log_BF (2nd - 1st)": zi2 - zi1}).to_csv(OUT / "6_sigma_path_bayes_factor.csv", index=False)
    fig, axs = plt.subplots(1, 2, figsize=(9, 3.2))
    axs[0].semilogx(sg, zi1, "-", label=LABEL); axs[0].semilogx(sg, zi2, "--", label=COMPARE["label"])
    axs[0].set_xlabel("σ'"); axs[0].set_ylabel("log Z(σ') (SMC, mean of the islands)"); axs[0].legend(fontsize=7)
    axs[1].semilogx(sg, zi2 - zi1, "k-"); axs[1].fill_between(sg, zi2 - zi1 - np.hypot(se1, se2), zi2 - zi1 + np.hypot(se1, se2), color="0.8")
    axs[1].axhline(0, color="0.5", lw=0.8)
    axs[1].set_xlabel("σ'"); axs[1].set_ylabel(f"log BF ({COMPARE['label']} vs {LABEL})")
    for a in axs: a.axvline(sigma, color="C3", ls=":", lw=0.8); a.grid(alpha=0.3)
    fig.tight_layout(); report.save_figure(fig, FIG / "6_model_comparison_sigma_path.png"); plt.show()
    display(cmp_tab.round(4))

## 7. HDF5 corner export (schema of cylinder.ipynb, `/corner/physical`) and previews

In [ ]:
start_theta = np.array(json.loads((RUN_IN / "start_point.json").read_text())["theta"], float) \
    if (RUN_IN / "start_point.json").exists() else None
H5 = {}
for name, res_ in (("smc", smc_res), ("amis", amis_res)):
    if res_ is None:
        continue
    def export(res_=res_, name=name):
        return smc.write_h5(OUT / f"{name}.h5", res_, prior, best_theta=theta_star, truth_theta=truth_theta,
                            start_theta=start_theta if start_theta is not None and prior.in_support(start_theta[None])[0] else None,
                            attrs={"run_in": str(RUN_IN), "mode": MODE, "network": str(nn_dir), "sigma_trpl": sigma,
                                   "sigma_sspl": np.nan if sigma_ss is None else sigma_ss, "label": LABEL})
    H5[name], t_ex = timeit_call(export, repeats=1, label=f"corner densities + HDF5 export ({name})"); log_timing(t_ex)
    with h5py.File(H5[name], "r") as f:
        fig = cp.plot_corner(f["corner/physical"], title=f"{RUN_IN.name}: {name.upper()} posterior, σ = {sigma:.4f}")
    report.save_figure(fig, FIG / f"7_corner_{name}_preview.png"); fig.savefig(FIG / f"7_corner_{name}_preview.pdf"); plt.show()
print({k: str(v) for k, v in H5.items()})

## 8. Timing table and summary

In [ ]:
timing = pd.DataFrame(TIMINGS)
timing["time"] = [format_mean_sd(m, s, int(n), "s") for m, s, n in zip(timing.mean_s, timing.std_s, timing.repeats)]
timing.to_csv(OUT / "8_timing.csv", index=False)
summary = {"run_in": str(RUN_IN), "mode": MODE, "label": LABEL, "network": nn_dir.name, "sigma_trpl": sigma,
           "sigma_sspl": sigma_ss, "sigma": sig_info,
           "nuts_reference": NUTS_REFERENCE, "references_of_run_in_valid": REF_OK,
           "smc": {"log_z": smc_res.log_evidence, "log_z_se": smc_res.log_evidence_se, "n_stages": si["n_stages"],
                   "evals": si["evals_total_points"], "calls": si["evals_total_calls"], "time_mean_s": t_smc.mean_s,
                   "time_sd_s": t_smc.std_s, "ess_kish": si["ess_weighted_kish"], "min_ess_bulk": float(diag_smc.ess_bulk.min()),
                   "max_r_hat": float(diag_smc.r_hat.max())},
           "amis": None if amis_res is None else {"log_z": amis_res.log_evidence, "log_z_se": amis_res.log_evidence_se,
                   "k_hat": ai["k_hat"], "ess_kish": amis_res.ess, "evals": ai["evals_total_points"], "time_mean_s": t_amis.mean_s,
                   "time_sd_s": t_amis.std_s},
           "cylinder_is": None if cyl_res is None else cyl_summary | {"time_mean_s": t_cyl.mean_s},
           "nuts": None if nuts_res is None else {"time": nuts_time, "evals": nuts_res.info.get("evals_total_points"),
                                                  "max_r_hat": float(diag_nuts.r_hat.max()), "min_ess_bulk": float(diag_nuts.ess_bulk.min())},
           "coverage_of_nuts_mass": None if cover is None else cover.to_dict("records"),
           "c2st_vs_nuts": None if c2 is None else c2.to_dict(),
           "max_quantile_deviation_vs_nuts_in_sd": None if dev is None else dev.to_dict("index"),
           "credible_intervals_smc": ci_smc.reset_index().to_dict("records"),
           "model_comparison": None if cmp_tab is None else cmp_tab.to_dict("records")}
report.save_json(summary, OUT / "summary.json")
print("written:", OUT)
if UPDATE_T8 and REF_OK and (RUN_IN / "11_T8_timing_table.csv").exists():      # SMC row of the T8 table of RUN_IN
    t8_ = bm.update_t8(RUN_IN, smc_dir=OUT)
    print("T8 table of RUN_IN updated:", t8_.step.iloc[-1])
display(timing[["label", "time", "n_evals", "note"]])
eff.round(3)